# Local model lifecycle

Read [lessons 02](../../docs/02-data-and-experiments.md) and
[03](../../docs/03-evaluation-and-promotion.md). Run all cells from top to bottom.
This notebook uses a temporary workspace and does not change the CLI registry.
The original `iris.ipynb` uses 80/20; this lifecycle uses 60/20/20.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from iris_mlops.data import prepare_data
from iris_mlops.workflow import (train_run, evaluate, promote, rollback, registry,
                      export_bundle, final_test, run_path, read_json)

workspace = TemporaryDirectory()
work = Path(workspace.name)
data_dir, state_dir = work / "data", work / "state"
frames = prepare_data(data_dir)
pd.DataFrame({name: frame.label.value_counts() for name, frame in frames.items()})


## Train candidates and inspect evidence

The scaler learns from the 90 training rows only. Change `C` to explore
regularization. We compare on validation data; test data remains unopened.


In [ ]:
candidates = {"weak": 0.000001, "baseline": 1.0, "candidate": 10.0}
reports = []
for name, C in candidates.items():
    train_run(data_dir, root=state_dir, C=C, run_id=name)
    gate = evaluate(name, root=state_dir)
    reports.append({"run_id": name, "C": C, "passed": gate["passed"],
                    "accuracy": gate["validation"]["accuracy"],
                    "macro_f1": gate["validation"]["macro_f1"]})
comparison = pd.DataFrame(reports).set_index("run_id")
display(comparison)
comparison[["accuracy", "macro_f1"]].plot.bar(ylim=(0, 1), rot=0, title="Validation scores")
plt.axhline(0.9, color="black", linestyle="--", label="Minimum")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
metadata = read_json(run_path("baseline", state_dir) / "run.json")
metadata


## A failed gate must leave production unchanged

Try to promote the weak candidate. Failure is expected and handled here so
Run All can continue. A threshold failure is a release decision, not a crash
that should be hidden by changing the policy.


In [ ]:
try:
    promote("weak", root=state_dir)
except ValueError as error:
    print(error)
assert registry(state_dir)["production"] is None


## Review and promote the baseline

Inspect its gate and confusion matrix before running the promotion cell.
Passing candidates need not replace production just because they tie its score.


In [ ]:
display(read_json(run_path("baseline", state_dir) / "gate.json"))
promote("baseline", root=state_dir)
registry(state_dir)


## Report the test score and export

Selection is now complete. The explicit test report is cached; do not use it
to choose another `C`. The export contains the scaler, classifier and review
evidence, without the training CSVs.


In [ ]:
display(final_test("baseline", root=state_dir))
bundle = export_bundle("baseline", work / "baseline-release", root=state_dir)
display([path.name for path in bundle.iterdir()])


## Exercise the HTTP contract without starting a separate server

The same app runs in Docker and both clouds. Check the species and version.


In [ ]:
from fastapi.testclient import TestClient
from iris_mlops.serve import create_app
from iris_mlops.cloud_common import SAMPLE

with TestClient(create_app(bundle)) as client:
    print(client.get("/health").json())
    response = client.post("/predict", json=SAMPLE)
    display(response.json())
    assert response.json()["model_version"] == "baseline"
    assert client.post("/predict", json={"instances": []}).status_code == 422


## Promote another passing version, then roll back

The second promotion re-evaluates against the current incumbent. Rollback is a
separate recorded decision. A running process would still need a restart.


In [ ]:
promote("candidate", root=state_dir)
rollback("baseline", root=state_dir)
pd.DataFrame(registry(state_dir)["history"])


## Check your understanding

- Why are training accuracy and test-based hyperparameter selection misleading?
- Which files identify the data, code and runtime of a run?
- What does changing the production pointer do to an already-running service?
- What additional evidence would you need before replacing a real model?

Temporary files live for this kernel session. Restart the kernel and Run All
for a fresh exercise; use the CLI lessons for persistent releases.
